# MultiTaxi benchmark

Run this notebook from the project's `app/` environment. One versioned JSON file stores each run, its final evaluation, and its convergence checkpoints.

QT is trained once per seed and reused as the baseline in each reward-machine row. RM, CRM, and HRM are trained for every configured reward machine.

In [ ]:
import fcntl
import hashlib
import json
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from multiprocessing import get_context

import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from scripts import train_qt_hrm_agent, train_qt
from src.config import Configuration
from src.models import RewardMachine, evaluate_agent

In [ ]:
@dataclass
class BenchmarkConfiguration(Configuration):
    training_seeds: tuple[int, ...] = tuple(range(42, 52))
    convergence_interval: int = 1000
    parallel_workers: int = 4
    rerun_completed: bool = False
    record_version: int = 6


BENCHMARK = BenchmarkConfiguration(
    n_training_episodes=200000,
    n_eval_episodes=200,
    eval_seed_base=20260720,
    multitaxi_grid_size=10,
    multitaxi_reward_shaping=False,
    video_fps=10,
)
TRAINING_SEEDS = list(BENCHMARK.training_seeds)
EVALUATION_SEEDS = BENCHMARK.eval_seed
if not TRAINING_SEEDS:
    raise ValueError('training_seeds cannot be empty')
if BENCHMARK.convergence_interval <= 0:
    raise ValueError('convergence_interval must be positive')
if BENCHMARK.parallel_workers <= 0:
    raise ValueError('parallel_workers must be positive')
if BENCHMARK.multitaxi_reward_shaping:
    raise ValueError('MultiTaxi reward shaping must be disabled for this benchmark')

MODES = (
    {'id': 'qt', 'name': 'QT', 'kind': 'qt', 'config': 'qt.yaml'},
    {'id': 'rm', 'name': 'RM', 'kind': 'qt', 'config': 'qt_rm.yaml'},
    {'id': 'crm', 'name': 'CRM', 'kind': 'qt', 'config': 'qt_crm.yaml'},
    {'id': 'hrm', 'name': 'HRM', 'kind': 'qt_hrm', 'config': 'qt_hrm.yaml'},
)
MODES_BY_ID = {mode['id']: mode for mode in MODES}
REWARD_MACHINES = (
    {'id': '3s', 'name': '3 states', 'file': 'rm_taxi_2p_3s.txt'},
    {'id': '4s', 'name': '4 states', 'file': 'rm_taxi_2p_4s.txt'},
    {'id': '9s', 'name': '9 states', 'file': 'rm_taxi_2p_9s.txt'},
)


def experiment_id(mode, reward_machine=None):
    return mode['id'] if reward_machine is None else f"{mode['id']}_{reward_machine['id']}"


def build_experiments():
    experiments = []
    for mode in MODES:
        reward_machines = (None,) if mode['id'] == 'qt' else REWARD_MACHINES
        for reward_machine in reward_machines:
            experiments.append({
                'id': experiment_id(mode, reward_machine),
                'mode_id': mode['id'],
                'mode': mode['name'],
                'kind': mode['kind'],
                'config': mode['config'],
                'reward_machine_id': None if reward_machine is None else reward_machine['id'],
                'reward_machine': None if reward_machine is None else reward_machine['name'],
                'rm_file': None if reward_machine is None else reward_machine['file'],
            })
    return tuple(experiments)


EXPERIMENTS = build_experiments()
EXPERIMENTS_BY_ID = {experiment['id']: experiment for experiment in EXPERIMENTS}
RESULTS_PATH = os.path.join(
    BENCHMARK.DATA_PATH,
    f'multitaxi_{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_benchmark_v{BENCHMARK.record_version}.json',
)
RESULTS_LOCK_PATH = f'{RESULTS_PATH}.lock'
BENCHMARK.VIDEO_PATH = os.path.join(
    BENCHMARK.VIDEO_PATH,
    f'multitaxi_{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_benchmark_v{BENCHMARK.record_version}',
)
os.makedirs(BENCHMARK.VIDEO_PATH, exist_ok=True)


def video_experiment_id(experiment):
    reward_machine_id = experiment['reward_machine_id'] or 'baseline'
    return f"{experiment['mode_id']}_{reward_machine_id}"


def benchmark_config(seed, experiment):
    mode = MODES_BY_ID[experiment['mode_id']]
    config = Configuration(yaml_config_path=mode['config'])
    config.set_seed(seed)
    config.exp_name = video_experiment_id(experiment)
    config.rm_file = experiment['rm_file']
    config.multitaxi_grid_size = BENCHMARK.multitaxi_grid_size
    config.multitaxi_reward_shaping = BENCHMARK.multitaxi_reward_shaping
    config.VIDEO_PATH = BENCHMARK.VIDEO_PATH
    config.video_fps = BENCHMARK.video_fps
    config.n_training_episodes = BENCHMARK.n_training_episodes
    config.n_eval_episodes = BENCHMARK.n_eval_episodes
    config.eval_seed = EVALUATION_SEEDS
    if config.multitaxi_reward_shaping:
        raise ValueError('MultiTaxi reward shaping must be disabled')
    if experiment['mode_id'] != 'qt' and not config.rm_file:
        raise ValueError(f"{experiment['id']} requires a reward machine file")
    return config


for reward_machine in REWARD_MACHINES:
    reward_machine_path = os.path.join(BENCHMARK.MODELS_PATH, reward_machine['file'])
    if not os.path.isfile(reward_machine_path):
        raise FileNotFoundError(reward_machine_path)
    RewardMachine(BENCHMARK, reward_machine['file'])


def reward_machine_spec(reward_machine):
    path = os.path.join(BENCHMARK.MODELS_PATH, reward_machine['file'])
    with open(path, 'rb') as file:
        checksum = hashlib.sha256(file.read()).hexdigest()
    return {**reward_machine, 'sha256': checksum}


def mode_spec(mode):
    experiment = next(experiment for experiment in EXPERIMENTS if experiment['mode_id'] == mode['id'])
    config = benchmark_config(BENCHMARK.seed, experiment)
    config.exp_name = experiment['id']
    config_path = os.path.join(config.CONFIGS_PATH, mode['config'])
    with open(config_path, encoding='utf-8') as file:
        yaml = file.read()
    return {
        'id': mode['id'],
        'name': mode['name'],
        'kind': mode['kind'],
        'config': mode['config'],
        'use_rm': config.use_rm,
        'use_crm': config.use_crm,
        'observation': config.multitaxi_observation_mode,
        'yaml': yaml,
        'resolved': {
            key: value
            for key, value in asdict(config).items()
            if key not in {'CONFIGS_PATH', 'DATA_PATH', 'MODELS_PATH', 'LOGS_PATH', 'VIDEO_PATH', 'eval_seed', 'seed', 'yaml_config_path'}
        },
    }


BENCHMARK_SPEC = {
    'version': BENCHMARK.record_version,
    'grid_size': BENCHMARK.multitaxi_grid_size,
    'reward_shaping': BENCHMARK.multitaxi_reward_shaping,
    'training_episodes': BENCHMARK.n_training_episodes,
    'evaluation_episodes': BENCHMARK.n_eval_episodes,
    'training_seeds': TRAINING_SEEDS,
    'evaluation_seeds': EVALUATION_SEEDS,
    'video_fps': BENCHMARK.video_fps,
    'convergence_interval': BENCHMARK.convergence_interval,
    'modes': [mode_spec(mode) for mode in MODES],
    'reward_machines': [reward_machine_spec(reward_machine) for reward_machine in REWARD_MACHINES],
    'experiments': [
        {
            'id': experiment['id'],
            'mode_id': experiment['mode_id'],
            'reward_machine_id': experiment['reward_machine_id'],
            'rm_file': experiment['rm_file'],
        }
        for experiment in EXPERIMENTS
    ],
}

print(f'Unique training runs: {len(EXPERIMENTS) * len(TRAINING_SEEDS)}')
print(f'Parallel seed workers: {min(BENCHMARK.parallel_workers, len(TRAINING_SEEDS))}')
print(f'Training episodes per run: {BENCHMARK.n_training_episodes}')
print(f'Evaluation episodes per run: {len(EVALUATION_SEEDS)}')
print(f'Results: {RESULTS_PATH}')

In [ ]:
METRIC_KEYS = (
    'successes', 'episodes', 'invalid_actions', 'mean_reward', 'reward_std',
    'successful_std', 'mean_successful_steps', 'worst_reward',
)


def normalize_metrics(metrics):
    normalized = {}
    for key in METRIC_KEYS:
        value = metrics[key]
        if isinstance(value, np.generic):
            value = value.item()
        normalized[key] = None if isinstance(value, float) and not np.isfinite(value) else value
    return normalized


def validate_metrics(metrics):
    if not isinstance(metrics, dict) or not set(METRIC_KEYS) <= metrics.keys():
        raise ValueError('Evaluation metrics do not match the experiment schema')
    for key in METRIC_KEYS:
        value = metrics[key]
        if value is None and key in {'successful_std', 'mean_successful_steps'}:
            continue
        if not isinstance(value, (int, float)) or not np.isfinite(value):
            raise ValueError('Evaluation metrics do not match the experiment schema')


def validate_run(run):
    required = {
        'experiment_id', 'mode_id', 'mode', 'kind', 'config',
        'reward_machine_id', 'reward_machine', 'rm_file', 'seed',
        'pipeline_seconds', 'metrics', 'convergence',
    }
    if not isinstance(run, dict) or not required <= run.keys():
        raise ValueError('Experiment runs do not match the v5 schema')
    experiment = EXPERIMENTS_BY_ID.get(run['experiment_id'])
    if experiment is None or any(
        run[key] != experiment[key]
        for key in ('mode_id', 'mode', 'kind', 'config', 'reward_machine_id', 'reward_machine', 'rm_file')
    ):
        raise ValueError('Experiment runs do not match the v5 schema')
    if not isinstance(run['seed'], int) or run['seed'] not in TRAINING_SEEDS:
        raise ValueError('Experiment runs do not match the v5 schema')
    if not isinstance(run['convergence'], list):
        raise ValueError('Experiment runs do not match the v5 schema')
    if run['metrics'] is None:
        if run['pipeline_seconds'] is not None:
            raise ValueError('Incomplete experiment runs cannot have a duration')
    else:
        validate_metrics(run['metrics'])
        if not isinstance(run['pipeline_seconds'], (int, float)) or run['pipeline_seconds'] < 0:
            raise ValueError('Experiment runs do not match the v5 schema')
    for checkpoint in run['convergence']:
        if (
            not isinstance(checkpoint, dict)
            or not isinstance(checkpoint.get('episode'), int)
            or not 0 < checkpoint['episode'] <= BENCHMARK.n_training_episodes
        ):
            raise ValueError('Convergence checkpoints do not match the v5 schema')
        validate_metrics(checkpoint.get('metrics'))
    if len({checkpoint['episode'] for checkpoint in run['convergence']}) != len(run['convergence']):
        raise ValueError('Experiment runs contain duplicate convergence checkpoints')


def run_key(run):
    return run['experiment_id'], run['seed']


@contextmanager
def results_file_lock(exclusive):
    with open(RESULTS_LOCK_PATH, 'a', encoding='utf-8') as lock_file:
        operation = fcntl.LOCK_EX if exclusive else fcntl.LOCK_SH
        fcntl.flock(lock_file.fileno(), operation)
        try:
            yield
        finally:
            fcntl.flock(lock_file.fileno(), fcntl.LOCK_UN)


def validate_runs(runs):
    if not isinstance(runs, list):
        raise ValueError('Experiment runs must be a list')
    for run in runs:
        validate_run(run)
    if len({run_key(run) for run in runs}) != len(runs):
        raise ValueError('Experiment records contain duplicate runs')


def read_runs():
    if not os.path.exists(RESULTS_PATH):
        return []
    with open(RESULTS_PATH, encoding='utf-8') as file:
        payload = json.load(file)
    if not isinstance(payload, dict) or payload.get('spec') != BENCHMARK_SPEC:
        raise ValueError(f'Results at {RESULTS_PATH} do not match the current benchmark specification.')
    runs = payload.get('runs')
    validate_runs(runs)
    return runs


def load_runs():
    with results_file_lock(exclusive=False):
        return read_runs()


def write_runs(runs):
    validate_runs(runs)
    temporary_path = f'{RESULTS_PATH}.{os.getpid()}.tmp'
    try:
        with open(temporary_path, 'w', encoding='utf-8') as file:
            json.dump({'spec': BENCHMARK_SPEC, 'runs': runs}, file, indent=2, allow_nan=False)
        os.replace(temporary_path, RESULTS_PATH)
    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)


def save_runs(runs):
    with results_file_lock(exclusive=True):
        write_runs(runs)


def upsert_run(run):
    validate_run(run)
    with results_file_lock(exclusive=True):
        runs = read_runs()
        runs = [stored_run for stored_run in runs if run_key(stored_run) != run_key(run)]
        runs.append(run)
        write_runs(runs)


def completed_run(run):
    return run['metrics'] is not None and any(
        checkpoint['episode'] == BENCHMARK.n_training_episodes
        for checkpoint in run['convergence']
    )


RUNNERS = {
    'qt': train_qt,
    'qt_hrm': train_qt_hrm_agent,
}


def video_path(experiment, seed):
    return os.path.join(
        BENCHMARK.VIDEO_PATH,
        f"{BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size}_{video_experiment_id(experiment)}_seed{seed}_video.gif",
    )


def new_run(experiment, seed):
    return {
        'experiment_id': experiment['id'],
        'mode_id': experiment['mode_id'],
        'mode': experiment['mode'],
        'kind': experiment['kind'],
        'config': experiment['config'],
        'reward_machine_id': experiment['reward_machine_id'],
        'reward_machine': experiment['reward_machine'],
        'rm_file': experiment['rm_file'],
        'seed': seed,
        'pipeline_seconds': None,
        'metrics': None,
        'convergence': [],
    }


def run_experiment(experiment, seed):
    key = (experiment['id'], seed)
    existing_run = next((run for run in load_runs() if run_key(run) == key), None)
    if (
        existing_run
        and completed_run(existing_run)
        and os.path.isfile(video_path(experiment, seed))
        and not BENCHMARK.rerun_completed
    ):
        return f"Skipping {experiment['mode']} {experiment['reward_machine'] or 'baseline'}, seed {seed}"

    run = new_run(experiment, seed)
    upsert_run(run)
    config = benchmark_config(seed, experiment)

    def progress_callback(episode, agent, env, get_propositions):
        if episode % BENCHMARK.convergence_interval and episode != BENCHMARK.n_training_episodes:
            return
        metrics = normalize_metrics(evaluate_agent(
            config, agent, get_propositions, env,
            seeds=EVALUATION_SEEDS, report=False, return_metrics=True,
        ))
        run['convergence'] = [
            checkpoint
            for checkpoint in run['convergence']
            if checkpoint['episode'] != episode
        ]
        run['convergence'].append({'episode': episode, 'metrics': metrics})
        run['convergence'].sort(key=lambda checkpoint: checkpoint['episode'])
        upsert_run(run)

    started = time.perf_counter()
    RUNNERS[experiment['kind']](config, progress_callback=progress_callback)
    final_checkpoint = next(
        (checkpoint for checkpoint in run['convergence'] if checkpoint['episode'] == BENCHMARK.n_training_episodes),
        None,
    )
    if final_checkpoint is None:
        raise RuntimeError('Training finished without a final evaluation checkpoint')
    run['pipeline_seconds'] = time.perf_counter() - started
    run['metrics'] = final_checkpoint['metrics']
    upsert_run(run)
    return (
        f"{experiment['mode']} {experiment['reward_machine'] or 'baseline'}, seed {seed}: "
        f"{run['metrics']['mean_reward']:.2f} reward, "
        f"{run['metrics']['mean_successful_steps']} mean successful steps"
    )


artifact_paths = [
    video_path(experiment, seed)
    for experiment in EXPERIMENTS
    for seed in TRAINING_SEEDS
]
if len(artifact_paths) != len(set(artifact_paths)):
    raise ValueError('Experiment video paths are not unique')

In [ ]:
for experiment in EXPERIMENTS:
    print(f"Starting {experiment['mode']} {experiment['reward_machine'] or 'baseline'}")
    with ProcessPoolExecutor(
        max_workers=min(BENCHMARK.parallel_workers, len(TRAINING_SEEDS)),
        mp_context=get_context('fork'),
    ) as executor:
        futures = [executor.submit(run_experiment, experiment, seed) for seed in TRAINING_SEEDS]
        for future in as_completed(futures):
            print(future.result())

In [ ]:
runs = [run for run in load_runs() if completed_run(run)]
if not runs:
    raise ValueError(f'No completed benchmark runs found in {RESULTS_PATH}')


def runs_for_condition(all_runs, mode, reward_machine):
    expected_reward_machine_id = None if mode['id'] == 'qt' else reward_machine['id']
    return [
        run
        for run in all_runs
        if run['mode_id'] == mode['id']
        and run['reward_machine_id'] == expected_reward_machine_id
    ]


def condition_summary(condition_runs, reward_machine, mode):
    rewards = np.asarray([run['metrics']['mean_reward'] for run in condition_runs])
    steps = np.asarray([
        run['metrics']['mean_successful_steps']
        for run in condition_runs
        if run['metrics']['mean_successful_steps'] is not None
    ])
    successes = sum(run['metrics']['successes'] for run in condition_runs)
    episodes = sum(run['metrics']['episodes'] for run in condition_runs)
    return {
        'reward_machine': reward_machine['name'],
        'mode': mode['name'],
        'runs': len(condition_runs),
        'mean_reward': float(rewards.mean()),
        'reward_std_across_runs': float(rewards.std()),
        'success_rate': successes / episodes if episodes else None,
        'mean_successful_steps': float(steps.mean()) if len(steps) else None,
    }


summary = []
for reward_machine in REWARD_MACHINES:
    for mode in MODES:
        condition_runs = runs_for_condition(runs, mode, reward_machine)
        if condition_runs:
            summary.append(condition_summary(condition_runs, reward_machine, mode))

display(summary)

In [ ]:
def make_metric_figure(metric_key, title, yaxis_title):
    figure = make_subplots(
        rows=len(REWARD_MACHINES),
        cols=len(MODES),
        row_titles=[reward_machine['name'] for reward_machine in REWARD_MACHINES],
        column_titles=[mode['name'] for mode in MODES],
        horizontal_spacing=0.04,
        vertical_spacing=0.08,
    )
    for row, reward_machine in enumerate(REWARD_MACHINES, start=1):
        for col, mode in enumerate(MODES, start=1):
            condition_runs = runs_for_condition(runs, mode, reward_machine)
            values = [
                run['metrics'][metric_key]
                for run in condition_runs
                if run['metrics'][metric_key] is not None
            ]
            if values:
                figure.add_trace(
                    go.Box(
                        y=values,
                        name=mode['name'],
                        boxpoints='all',
                        jitter=0.25,
                        pointpos=0,
                        marker={'size': 5},
                        showlegend=False,
                        hovertemplate='Value: %{y:.2f}<extra></extra>',
                    ),
                    row=row,
                    col=col,
                )
            figure.update_xaxes(showticklabels=False, row=row, col=col)
            figure.update_yaxes(
                title_text=yaxis_title if col == 1 else None,
                row=row,
                col=col,
            )
    figure.update_layout(
        title=(
            f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} {title} '
            '(QT is the shared baseline across rows)'
        ),
        template='plotly_white',
        height=850,
        width=1400,
        showlegend=False,
        margin={'l': 120, 'r': 30, 't': 100, 'b': 40},
    )
    return figure


reward_figure = make_metric_figure(
    'mean_reward',
    'final evaluation reward',
    'Mean environment reward',
)
reward_figure.show()

steps_figure = make_metric_figure(
    'mean_successful_steps',
    'steps to solve successful episodes',
    'Mean steps',
)
steps_figure.show()

In [ ]:
def convergence_series(condition_runs):
    values_by_episode = {}
    for run in condition_runs:
        for checkpoint in run['convergence']:
            values_by_episode.setdefault(checkpoint['episode'], {})[run['seed']] = (
                checkpoint['metrics']['mean_reward']
            )
    episodes = [
        episode
        for episode, values in sorted(values_by_episode.items())
        if set(values) == set(TRAINING_SEEDS)
    ]
    values = [
        [values_by_episode[episode][seed] for seed in TRAINING_SEEDS]
        for episode in episodes
    ]
    return episodes, values


convergence_figure = make_subplots(
    rows=len(REWARD_MACHINES),
    cols=1,
    row_titles=[reward_machine['name'] for reward_machine in REWARD_MACHINES],
    shared_xaxes=True,
    vertical_spacing=0.08,
)
for row, reward_machine in enumerate(REWARD_MACHINES, start=1):
    for mode in MODES:
        condition_runs = runs_for_condition(runs, mode, reward_machine)
        episodes, values = convergence_series(condition_runs)
        if not episodes:
            continue
        convergence_figure.add_trace(
            go.Scatter(
                x=episodes,
                y=[np.mean(episode_values) for episode_values in values],
                error_y={
                    'type': 'data',
                    'array': [np.std(episode_values) for episode_values in values],
                    'visible': True,
                },
                mode='lines+markers',
                name=mode['name'],
                showlegend=row == 1,
            ),
            row=row,
            col=1,
        )
    convergence_figure.update_yaxes(
        title_text='Mean environment reward',
        row=row,
        col=1,
    )

convergence_figure.update_layout(
    title=f'MultiTaxi {BENCHMARK.multitaxi_grid_size}x{BENCHMARK.multitaxi_grid_size} reward convergence',
    xaxis_title='Training episodes',
    template='plotly_white',
    height=850,
    width=1200,
)
convergence_figure.show()